## Assignment - 2 : Building a SLR Model for Corruption Perception

## #1 Importing the Necessary Libraries
We begin by importing the standard libraries required for data manipulation, visualization, and statistical modeling.

* Pandas: Used for data structures and CSV file handling.

* Numpy: Essential for numerical operations and array processing.

* Matplotlib: Used for generating scatter plots to observe data trends.

* Statsmodels: Specifically chosen for this assignment to perform detailed statistical inference, including p-values and confidence intervals.

* Sklearn: Used here for splitting the data and calculating performance metrics.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## #2 Loading the Dataset
We load the country.csv file into a DataFrame. This dataset captures the socioeconomic relationship between income inequality and perceived transparency across 20 nations.

In [21]:
# Loading the dataset
df = pd.read_csv('country.csv')

# Displaying the first five observations
df.head()

,Country,Corruption_Index,Gini_Index
0,Hong Kong,77,53.7
1,South Korea,53,30.2
2,China,40,46.2
3,Italy,47,32.7
4,Mongolia,38,36.5


## #3 Dataset Exploration
Before modeling, it is vital to understand the dimensions and quality of our data.

In [22]:
# Finding Number of Rows and Columns
print("Dataset Shape:", df.shape)

# Displaying Column Names
print("Columns:", df.columns.tolist())

# Checking for Missing Values
print("Missing Values:\n" , df.isna().sum())

Dataset Shape: (20, 3)
Columns: ['Country', 'Corruption_Index', 'Gini_Index']
Missing Values:
 Country             0
Corruption_Index    0
Gini_Index          0
dtype: int64


Observation: The dataset contains 20 observations and 2 primary columns. No missing values were detected, ensuring data integrity for the regression analysis.

## #4 Defining Independent and Dependent Variables
In this study, we aim to determine if income inequality influences how corruption is perceived.

Independent Variable (X): Gini Index (Predictor)

Dependent Variable (y): Corruption Perception Index (Target)

In [23]:
X = df["Gini_Index"]      # Independent variable
y = df["Corruption_Index"]       # Dependent variable

## #5 Splitting the Data for Training and Testing
To ensure the model is rigorous, we separate the data into a Training set and a Testing set. This allows us to train the model on one portion of the data and validate its accuracy on another portion it has never seen before.

In [24]:
# Splitting the data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## #6 Adding the Intercept Term
In Simple Linear Regression, we must explicitly add a constant term to our independent variable when using the statsmodels library. This term represents the y-intercept $b_0$, ensuring that the regression line does not erroneously pass through the origin (0,0) unless the data dictates it.

In [26]:
# Adding the constant to training and testing sets
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

## #7 Developing the Simple Linear Regression Model
We now develop the model of the form $Y = b_0 + b_1X$ using the Ordinary Least Squares (OLS) method on our training data

In [27]:
# Building and fitting the model on training data
model = sm.OLS(y_train, X_train_sm).fit()

## #8 Printing and Interpreting Coefficients
The coefficients define the mathematical relationship between our two variables.

In [32]:
# Printing Intercept and Slope
b0 = model.params.iloc[0]
b1 = model.params.iloc[1]

print("Intercept (b0):", b0)
print("Slope (b1):", b1)

Intercept (b0): 107.57451662500108
Slope (b1): -1.2581953454154255


* Intercept ($b_0$): This represents the theoretical Corruption Perception Index score if the Gini Index were zero.
* Slope ($b_1$): This indicates the predicted change in the Corruption Perception Index for every one-unit increase in the Gini Index.

## #9 The Regression Equation
Based on the model parameters, our regression equation is clearly defined as:$$\text{Corruption Perception Index} = 107.57451662500108 + (-1.2581953454154255 \times \text{Gini Index})$$

## #10 Meaning of the Slope in Context
The slope ($b_1$) quantifies the impact of income inequality on perceived corruption. A negative slope implies that as income inequality increases, the Corruption Perception Index tends to decrease, signifyng that higher inequality is associated with higher perceived corruption.

## #11 Model Evaluation (Test Set Performance)
To establish the model's reliability, we evaluate it using the test set. We calculate the $R^2$ score, Mean Absolute Error (MAE), and Mean Squared Error (MSE) to see how closely the predictions match reality.

In [39]:
# Generate predictions for the test set
y_pred_test = model.predict(X_test_sm)

# Calculate metrics
test_r2 = r2_score(y_test, y_pred_test)
test_mae = mean_absolute_error(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)

print(f"Test R² Score: {test_r2:.4f}")
print(f"Test Mean Absolute Error: {test_mae:.2f}")
print(f"Test Mean Squared Error: {test_mse:.2f}")

Test R² Score: -0.2121
Test Mean Absolute Error: 19.24
Test Mean Squared Error: 377.27


Interpretation: The $R^2$ value indicates the proportion of the variation in the Corruption Perception Index that is explained by the Gini Index on unseen data. Note that in a very small dataset ($n=20$), the test set consists of only 4 observations, which leads to volatile metrics like a negative $R^2$ because an outlier is present in the test sample.

## #12 Hypothesis Testing for $b_1$
We test the significance of the relationship at the $\alpha = 0.1$ significance level.
* Null Hypothesis ($H_0$): $b_1 = 0$ (The Gini Index has no effect on Corruption Perception).
* Alternative Hypothesis ($H_1$): $b_1 \neq 0$ (There is a significant relationship).

In [35]:
# Extracting the p-value for the Gini Index
p_value = model.pvalues.iloc[1]
print("p-value for b1:", p_value)

p-value for b1: 0.053011305465006396


Decision: Since the p-value (0.053) is less than our alpha of 0.1, we reject the Null Hypothesis. This concludes that there is a statistically significant relationship between income inequality and perceived corruption.

## #13 95% Confidence Interval for $b_1$
This range provides the interval in which we are 95% confident the true population slope lies.

In [37]:
# Calculating Confidence Intervals
conf_int = model.conf_int(alpha=0.05) # 95% CI
print("95% Confidence Interval for b1:\n", conf_int.loc['Gini_Index'])

95% Confidence Interval for b1:
 0   -2.535137
1    0.018746
Name: Gini_Index, dtype: float64


## #14 Prediction for Gini Index = 31
We use our model to estimate the transparency level for a country with a Gini Index score of 31.

In [38]:
# Predicting for Gini Index = 31
prediction = model.predict([1, 31])
print("Predicted Corruption Perception Index for Gini Index 31:", prediction[0])

Predicted Corruption Perception Index for Gini Index 31: 68.57046091712289


## #15 Final Conclusion
The Simple Linear Regression model provides a data-driven framework to understand the link between economic distribution and institutional transparency. By utilizing a Train/Test split, we have applied industry-standard validation practices to ensure the model is evaluated on unseen data.
Key takeaways from the analysis:
* Statistical Significance: The p-value of 0.053 confirms the relationship is significant at the $\alpha = 0.1$ level. This means we have sufficient evidence to reject the null hypothesis ($H_0$) and conclude that the Gini Index significantly impacts corruption perception.
* Inverse Relationship: The negative slope ($b_1 \approx -1.26$) confirms an inverse relationship: as income inequality increases, the Corruption Perception Index score drops, indicating higher levels of perceived corruption.
* The Baseline: The intercept ($b_0 \approx 107.57$) represents the theoretical transparency score if a country had perfect income equality (Gini Index of 0).
* Model Reliability: While the small test set size ($n=4$) can lead to volatile metrics like a negative $R^2$ score, the consistent significance of the slope and the low average prediction error (MAE) suggest the model is a meaningful tool for socioeconomic analysis.


Final Assessment: The model fits the underlying trend of the data well. It successfully quantifies how income distribution (Gini Index) serves as a reliable predictor for a nation's perceived transparency (Corruption Perception Index).